In [17]:
# ============================================================
# TASK 03 — MODEL EVALUATION & HYPERPARAMETER TUNING
# Titanic Dataset + Logistic Regression + GridSearchCV
# ============================================================

# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
# Removed zipfile and os as we'll directly load train.csv

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# ============================================================
# 2. LOAD TITANIC DATASET
# ============================================================

# Load the training data which contains the 'Survived' column
df = pd.read_csv("train.csv")

print("=" * 60)
print("TITANIC DATASET")
print("=" * 60)
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

# ============================================================
# 3. SELECT FEATURES
# ============================================================

features = ['Pclass', 'Sex', 'Age', 'Fare']

df = df[['Survived'] + features].copy()

# Handle missing Age values
df['Age'] = df['Age'].fillna(df['Age'].median())

# Convert Sex into numerical values
df['Sex'] = df['Sex'].map({
    'male': 0,
    'female': 1
})

# Remove any remaining missing values
df = df.dropna()

print("\nMissing values after preprocessing:")
print(df.isnull().sum())

# ============================================================
# 4. DEFINE X AND y
# ============================================================

X = df[features]
y = df['Survived']

# ============================================================
# 5. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

# ============================================================
# 6. ORIGINAL LOGISTIC REGRESSION MODEL
# ============================================================

original_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

original_model.fit(X_train, y_train)

# Predictions
y_pred_original = original_model.predict(X_test)

# ============================================================
# 7. ORIGINAL MODEL EVALUATION
# ============================================================

original_accuracy = accuracy_score(y_test, y_pred_original)
original_precision = precision_score(y_test, y_pred_original)
original_recall = recall_score(y_test, y_pred_original)
original_f1 = f1_score(y_test, y_pred_original)

print("\n" + "=" * 60)
print("ORIGINAL MODEL — CLASSIFICATION REPORT")
print("=" * 60)

print(classification_report(y_test, y_pred_original))

print("Original Accuracy :", round(original_accuracy, 4))
print("Original Precision:", round(original_precision, 4))
print("Original Recall   :", round(original_recall, 4))
print("Original F1-Score :", round(original_f1, 4))

# ============================================================
# 8. HYPERPARAMETER TUNING USING GRIDSEARCHCV
# ============================================================

# Tune TWO hyperparameters:
# 1. C
# 2. solver

param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__solver': ['liblinear', 'lbfgs']
}

grid_search = GridSearchCV(
    estimator=original_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# ============================================================
# 9. BEST PARAMETERS
# ============================================================

print("\n" + "=" * 60)
print("GRIDSEARCHCV RESULTS")
print("=" * 60)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation F1 Score:",
      round(grid_search.best_score_, 4))

# ============================================================
# 10. TUNED MODEL
# ============================================================

tuned_model = grid_search.best_estimator_

y_pred_tuned = tuned_model.predict(X_test)

# ============================================================
# 11. TUNED MODEL EVALUATION
# ============================================================

tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
tuned_precision = precision_score(y_test, y_pred_tuned)
tuned_recall = recall_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned)

print("\n" + "=" * 60)
print("TUNED MODEL — CLASSIFICATION REPORT")
print("=" * 60)

print(classification_report(y_test, y_pred_tuned))

print("Tuned Accuracy :", round(tuned_accuracy, 4))
print("Tuned Precision:", round(tuned_precision, 4))
print("Tuned Recall   :", round(tuned_recall, 4))
print("Tuned F1-Score :", round(tuned_f1, 4))

# ============================================================
# 12. BEFORE VS AFTER COMPARISON
# ============================================================

comparison = pd.DataFrame({
    'Metric': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1-Score'
    ],
    'Original Model': [
        original_accuracy,
        original_precision,
        original_recall,
        original_f1
    ],
    'Tuned Model': [
        tuned_accuracy,
        tuned_precision,
        tuned_recall,
        tuned_f1
    ]
})

# Round values for easier reading
comparison['Original Model'] = comparison['Original Model'].round(4)
comparison['Tuned Model'] = comparison['Tuned Model'].round(4)

print("\n" + "=" * 60)
print("BEFORE VS AFTER COMPARISON")
print("=" * 60)

print(comparison.to_string(index=False))

# ============================================================
# 13. CALCULATE IMPROVEMENT
# ============================================================

print("\n" + "=" * 60)
print("IMPROVEMENT AFTER TUNING")
print("=" * 60)

for metric, original, tuned in zip(
    ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    [original_accuracy, original_precision, original_recall, original_f1],
    [tuned_accuracy, tuned_precision, tuned_recall, tuned_f1]
):
    improvement = tuned - original
    print(f"{metric}: {improvement:+.4f}")

# ============================================================
# 14. OVERALL RESULT
# ============================================================

print("\n" + "=" * 60)
print("FINAL RESULT")
print("=" * 60)

if tuned_f1 > original_f1:
    print("Hyperparameter tuning IMPROVED the F1-score.")
elif tuned_f1 < original_f1:
    print("Hyperparameter tuning DECREASED the F1-score.")
else:
    print("Hyperparameter tuning produced the SAME F1-score.")

print("\nBest Hyperparameters:")
print(grid_search.best_params_)

print("\nTask completed successfully!")

TITANIC DATASET
Dataset shape: (891, 12)

First 5 rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  